In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import joblib
chunks = pd.read_csv('../data/accepted_2007_to_2018Q4.csv', chunksize=100000, parse_dates=['issue_d'], low_memory=False)
df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_34413/387980389.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_34413/387980389.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_filtered = pd.concat([c[(c['issue_d'] >= '2013-01-01') & (c['issue_d'] <= '2018-12-31')] for c in chunks])
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_34413/387980389.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please

In [2]:
import os

In [3]:
os.makedirs("data_splits", exist_ok=True)

def clean_series(s: pd.Series) -> pd.Series:
    if s.dtype != "object":
        return s
    def clean_val(v):
        if pd.isna(v):
            return None
        if isinstance(v, (bytes, bytearray)):
            return v.decode("utf-8", errors="replace")
        return str(v)
    return s.map(clean_val)

In [4]:
df_filtered["quarter"] = df_filtered["issue_d"].dt.to_period("Q")

In [5]:
import joblib
model = joblib.load("../api/models/xgbmodel1.pkl")

In [6]:
features = model.feature_names_in_

In [7]:
features

array(['funded_amnt', 'term', 'int_rate', 'installment', 'emp_length',
       'home_ownership', 'annual_inc', 'verification_status',
       'pymnt_plan', 'purpose', 'zip_code', 'addr_state', 'dti',
       'delinq_2yrs', 'inq_last_6mths', 'mths_since_last_delinq',
       'mths_since_last_record', 'pub_rec', 'revol_bal', 'revol_util',
       'total_acc', 'initial_list_status', 'out_prncp_inv', 'policy_code',
       'application_type', 'annual_inc_joint', 'dti_joint',
       'verification_status_joint', 'total_rev_hi_lim', 'inq_fi',
       'total_cu_tl', 'inq_last_12m', 'acc_open_past_24mths',
       'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq',
       'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl',
       'num_il_tl', 'num_op_rev_tl', 'num_rev_accts',
       'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_op_past_12m

In [8]:
df = df_filtered[features]
df.head()

,funded_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,pymnt_plan,purpose,...,total_bal_ex_mort,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts
0,3600.0,36 months,13.99,123.03,10+ years,MORTGAGE,55000.0,Not Verified,n,debt_consolidation,...,7746.0,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,24700.0,36 months,11.99,820.28,10+ years,MORTGAGE,65000.0,Not Verified,n,small_business,...,39475.0,79300.0,24667.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,20000.0,60 months,10.78,432.66,10+ years,MORTGAGE,63000.0,Not Verified,n,home_improvement,...,18696.0,6200.0,14877.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,35000.0,60 months,14.85,829.90,10+ years,MORTGAGE,110000.0,Source Verified,n,debt_consolidation,...,52226.0,62500.0,18000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10400.0,60 months,22.45,289.91,3 years,MORTGAGE,104433.0,Source Verified,n,major_purchase,...,95768.0,20300.0,88097.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
def col_to_num(df, amb_cols):
    for c in amb_cols:
        if c in ['emp_length', 'term']:
            df[c] = df[c].str.extract('(\d+)').astype('float64')
        df[c] = df[c].astype('float64')
    return df
# df2 = col_to_num(df, amb_cols)
# features['verification_status_joint'] = features['verification_status_joint'].map({'Not Verified': 0, 'Source Verified': 1, 'Verified': 2})

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler = StandardScaler()
label_encoder = LabelEncoder()

In [11]:
def scaler_and_encode(df, cat_labels):
    # Scale numerical features
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        df[col] = scaler.fit_transform(df[[col]])
    # Encode categorical features
    for col in cat_labels: # Exclude 'targets' from encoding
        df[col] = label_encoder.fit_transform(df[col].astype(str))
    return df

In [12]:
def pipeline(df, model_path):
    model = joblib.load(model_path)
    data = df.copy()
    X = data[model.feature_names_in_]
    df = col_to_num(X, ['emp_length', 'term'])
    df['verification_status_joint'] = df['verification_status_joint'].map({
        'Not Verified': 0, 'Source Verified': 1, 'Verified': 2})
    cat_labels = df.select_dtypes(include=['object']).columns
    df_scaled = scaler_and_encode(df, cat_labels)
    pred = model.predict_proba(df_scaled)
    data['prob_default']= pred[:,1]
    return data

In [13]:
X = df.head(10)
data_test = pipeline(X, "../api/models/xgbmodel1.pkl")

/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count
/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/Users/josephineamponsah/Documents/credit-scoring-nlp-dl/.venv/lib/python3.11/sit

In [14]:
data_test

,funded_amnt,term,int_rate,installment,emp_length,home_ownership,annual_inc,verification_status,pymnt_plan,purpose,...,total_bc_limit,total_il_high_credit_limit,revol_bal_joint,sec_app_inq_last_6mths,sec_app_mort_acc,sec_app_open_acc,sec_app_revol_util,sec_app_open_act_il,sec_app_num_rev_accts,prob_default
0,3600.0,36 months,13.99,123.03,10+ years,MORTGAGE,55000.0,Not Verified,n,debt_consolidation,...,2400.0,13734.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.438361
1,24700.0,36 months,11.99,820.28,10+ years,MORTGAGE,65000.0,Not Verified,n,small_business,...,79300.0,24667.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.460806
2,20000.0,60 months,10.78,432.66,10+ years,MORTGAGE,63000.0,Not Verified,n,home_improvement,...,6200.0,14877.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.375772
3,35000.0,60 months,14.85,829.90,10+ years,MORTGAGE,110000.0,Source Verified,n,debt_consolidation,...,62500.0,18000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.976332
4,10400.0,60 months,22.45,289.91,3 years,MORTGAGE,104433.0,Source Verified,n,major_purchase,...,20300.0,88097.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.643566
5,11950.0,36 months,13.44,405.18,4 years,RENT,34000.0,Source Verified,n,debt_consolidation,...,9400.0,4000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.532188
6,20000.0,36 months,9.17,637.58,10+ years,MORTGAGE,180000.0,Not Verified,n,debt_consolidation,...,31500.0,46452.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.341144
7,20000.0,36 months,8.49,631.26,10+ years,MORTGAGE,85000.0,Not Verified,n,major_purchase,...,14500.0,36144.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.295290
8,10000.0,36 months,6.49,306.45,6 years,RENT,85000.0,Not Verified,n,credit_card,...,16400.0,30799.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.274202
9,8000.0,36 months,11.48,263.74,10+ years,MORTGAGE,42000.0,Not Verified,n,credit_card,...,17000.0,135513.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.387778


In [15]:
full_dfs = pipeline(df_filtered, "../api/models/xgbmodel1.pkl")

/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_34413/519356066.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].str.extract('(\d+)').astype('float64')
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_34413/519356066.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[c] = df[c].astype('float64')
/var/folders/3f/9mw8q0xn58v4947ln1hhc4mr0000gn/T/ipykernel_34413/519356066.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.


In [16]:
df_filtered.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2164766 entries, 0 to 2260698
Columns: 152 entries, id to quarter
dtypes: datetime64[ns](1), float64(113), object(37), period[Q-DEC](1)
memory usage: 2.5+ GB


In [17]:
full_dfs.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2164766 entries, 0 to 2260698
Columns: 153 entries, id to prob_default
dtypes: datetime64[ns](1), float32(1), float64(113), object(37), period[Q-DEC](1)
memory usage: 2.5+ GB


In [19]:
for q, grp in full_dfs.groupby("quarter"):
    for col in grp.select_dtypes(include=["object"]).columns:
        grp[col] = clean_series(grp[col])
    fname = f"data_splits/data_{q}.parquet"
    grp.to_parquet(fname, index=False, compression="snappy")
